In [ ]:
# ---- Setup: get PlantDoc, cache it at 224x224, find checkpoints (Internet ON) ----
import glob, json, os, subprocess, sys, time
import numpy as np, torch
from concurrent.futures import ThreadPoolExecutor
from PIL import Image

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime"], check=False)
OUT = "/kaggle/working"
PD = f"{OUT}/PlantDoc-Dataset"
if not os.path.isdir(PD):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/pratikkayal/PlantDoc-Dataset", PD], check=True)

PV_CLASSES = ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy',
 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot',
 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)',
 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight',
 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew',
 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight',
 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Spot',
 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Tomato_mosaic_virus', 'Tomato___healthy']
MAP = {  # PlantDoc folder -> PlantVillage class
 "Apple Scab Leaf": "Apple___Apple_scab", "Apple leaf": "Apple___healthy", "Apple rust leaf": "Apple___Cedar_apple_rust",
 "Bell_pepper leaf spot": "Pepper,_bell___Bacterial_spot", "Bell_pepper leaf": "Pepper,_bell___healthy",
 "Blueberry leaf": "Blueberry___healthy", "Cherry leaf": "Cherry_(including_sour)___healthy",
 "Corn Gray leaf spot": "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot", "Corn leaf blight": "Corn_(maize)___Northern_Leaf_Blight",
 "Corn rust leaf": "Corn_(maize)___Common_rust_", "Peach leaf": "Peach___healthy",
 "Potato leaf early blight": "Potato___Early_blight", "Potato leaf late blight": "Potato___Late_blight",
 "Raspberry leaf": "Raspberry___healthy", "Soyabean leaf": "Soybean___healthy", "Squash Powdery mildew leaf": "Squash___Powdery_mildew",
 "Strawberry leaf": "Strawberry___healthy", "Tomato Early blight leaf": "Tomato___Early_blight",
 "Tomato Septoria leaf spot": "Tomato___Septoria_leaf_spot", "Tomato leaf bacterial spot": "Tomato___Bacterial_spot",
 "Tomato leaf late blight": "Tomato___Late_blight", "Tomato leaf mosaic virus": "Tomato___Tomato_mosaic_virus",
 "Tomato leaf yellow virus": "Tomato___Tomato_Yellow_Leaf_Curl_Virus", "Tomato leaf": "Tomato___healthy",
 "Tomato mold leaf": "Tomato___Leaf_Mold", "Tomato two spotted spider mites leaf": "Tomato___Spider_mites Two-spotted_spider_mite",
 "grape leaf black rot": "Grape___Black_rot", "grape leaf": "Grape___healthy",
}
PD_CLASSES = sorted(MAP)                      # 28 PlantDoc classes (fine-tuning head order)
PV_IDX = [PV_CLASSES.index(MAP[c]) for c in PD_CLASSES]

def load(p):
    try:
        return np.asarray(Image.open(p).convert("RGB").resize((224, 224), Image.BILINEAR))
    except Exception:
        return None

items = [(split, p, PD_CLASSES.index(d)) for split in ["train", "test"] for d in sorted(os.listdir(f"{PD}/{split}"))
         if d in MAP for p in sorted(glob.glob(f"{PD}/{split}/{d}/*"))]
with ThreadPoolExecutor(os.cpu_count()) as ex:
    imgs = list(ex.map(load, [p for _, p, _ in items]))
ok = [i for i, im in enumerate(imgs) if im is not None]
U8 = np.stack([imgs[i] for i in ok]); Y = np.array([items[i][2] for i in ok]); SPLIT = np.array([items[i][0] for i in ok])
FILES = np.array([os.path.relpath(items[i][1], PD) for i in ok])
# fixed 10% stratified validation split carved from PlantDoc train (for checkpoint selection during fine-tuning)
rng = np.random.default_rng(0); SUB = SPLIT.copy()
for c in range(len(PD_CLASSES)):
    tr = np.where((SPLIT == "train") & (Y == c))[0]
    if len(tr) >= 5:
        SUB[rng.choice(tr, max(1, round(0.1 * len(tr))), replace=False)] = "val"
np.savez(f"{OUT}/plantdoc_224.npz", u8=U8, y=Y, split=SUB, files=FILES, classes=np.array(PD_CLASSES), pv_idx=np.array(PV_IDX))
print(f"PlantDoc: {len(U8)} images ({len(items) - len(ok)} unreadable) | train {np.sum(SUB == 'train')}, "
      f"val {np.sum(SUB == 'val')}, test {np.sum(SUB == 'test')} | {len(PD_CLASSES)} classes")

RUNS = {"mobilenetv3_small_100": ["0", "1_full", "2"], "mobilevit_xxs": ["0", "1", "2"]}
CKPT = {}
for m, seeds in RUNS.items():
    for s in seeds:
        hits = glob.glob(f"/kaggle/input/**/{m}_final_seed{s}.pt", recursive=True)
        assert hits, f"{m}_final_seed{s}.pt not found - add the Phase 2 and rerun notebook outputs as inputs."
        CKPT[(m, s[0])] = hits[0]
print(len(CKPT), "PlantVillage checkpoints found")


In [ ]:
# ---- Part 1: cross-domain test. PlantVillage-trained models on all PlantDoc images, no training ----
import timm, pandas as pd, onnxruntime as ort
from sklearn.metrics import f1_score, recall_score

dev = "cuda" if torch.cuda.is_available() else "cpu"
crop = lambda name: name.split("___")[0]
Y_PV = np.array(PV_IDX)[Y]                              # PlantDoc labels expressed as PlantVillage class indices
MASK = np.full(38, -np.inf); MASK[sorted(set(PV_IDX))] = 0  # restrict predictions to classes that exist in PlantDoc

def norm_input(u8, m):
    c = timm.models.get_pretrained_cfg(m).to_dict()
    return ((u8.astype(np.float32) / 255).transpose(0, 3, 1, 2) - np.array(c["mean"], np.float32).reshape(1, 3, 1, 1)) \
        / np.array(c["std"], np.float32).reshape(1, 3, 1, 1)

def cross_metrics(logits, sel):
    y = Y_PV[sel]; lg = logits[sel]; p38 = lg.argmax(1); p = (lg + MASK).argmax(1)
    return {"acc_38way": float((p38 == y).mean()), "acc": float((p == y).mean()),
            "macro_f1": float(f1_score(y, p, average="macro", labels=sorted(set(y)))),
            "crop_acc": float(np.mean([crop(PV_CLASSES[a]) == crop(PV_CLASSES[b]) for a, b in zip(p38, y)]))}

zs_rows, per_class = [], {}
subsets = {"all PlantDoc": np.ones(len(Y), bool), "PlantDoc test split": SPLIT == "test"}
for (m, s), ck in CKPT.items():
    model = timm.create_model(m, pretrained=False, num_classes=38); model.load_state_dict(torch.load(ck, map_location="cpu"))
    model.to(dev).eval(); x = norm_input(U8, m)
    with torch.no_grad():
        lg = np.concatenate([model(torch.from_numpy(x[i:i + 128]).to(dev)).float().cpu().numpy() for i in range(0, len(x), 128)])
    for name, sel in subsets.items():
        zs_rows.append({"model": m, "seed": int(s), "precision": "fp32 (PyTorch)", "subset": name, **cross_metrics(lg, sel)})
    per_class[(m, s)] = recall_score(Y_PV, (lg + MASK).argmax(1), average=None, labels=PV_IDX, zero_division=0)
    print(f"{m} seed {s}: " + json.dumps({k: round(v, 4) for k, v in zs_rows[-2].items() if isinstance(v, float)}), flush=True)

# Quantized ONNX models from the earlier notebooks, if their outputs are attached (optional)
for f in sorted(glob.glob("/kaggle/input/**/*_seed*_int8*.onnx", recursive=True) + glob.glob("/kaggle/input/**/*_seed*_fp16.onnx", recursive=True)):
    name = os.path.basename(f); m = "mobilevit_xxs" if "mobilevit" in name else "mobilenetv3_small_100"
    sess = ort.InferenceSession(f, providers=["CPUExecutionProvider"]); x = norm_input(U8, m)
    lg = np.concatenate([sess.run(None, {"input": x[i:i + 64]})[0] for i in range(0, len(x), 64)])
    seed = int(name.split("_seed")[1][0]); variant = name.split(f"_seed{seed}_")[1].replace(".onnx", "")
    zs_rows.append({"model": m, "seed": seed, "precision": variant, "subset": "all PlantDoc", **cross_metrics(lg, np.ones(len(Y), bool))})
    print(f"{name}: acc {zs_rows[-1]['acc']:.4f}", flush=True)

zs = pd.DataFrame(zs_rows); zs.to_csv(f"{OUT}/plantdoc_crossdomain.csv", index=False)
pc = pd.DataFrame({f"{m}_seed{s}": v for (m, s), v in per_class.items()}, index=PD_CLASSES)
pc.to_csv(f"{OUT}/plantdoc_crossdomain_per_class_recall.csv")
print("\nCross-domain summary (FP32, mean ± std over seeds):")
print(zs[zs.precision.str.startswith("fp32")].groupby(["subset", "model"])[["acc_38way", "acc", "macro_f1", "crop_acc"]]
      .agg(["mean", "std"]).round(4).to_string())


In [ ]:
%%writefile /kaggle/working/plantdoc_ft.py
"""
Fine-tune one model on PlantDoc train, select the checkpoint on the PlantDoc validation carve-out, test once on PlantDoc test.
--init imagenet      : timm ImageNet-pretrained weights, new 28-class head
--init plantvillage  : our PlantVillage checkpoint; the 28-class head is initialised from the matching PlantVillage head rows
Hyperparameters: each model's winning PlantVillage configuration (lr, wd, batch, dropout, label smoothing, cosine schedule).
"""
import argparse, json, math, os, random, time
import numpy as np, timm, torch, torch.nn as nn
from PIL import Image
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

CFG = {"mobilenetv3_small_100": {"lr": 2.0436570225942804e-4, "weight_decay": 3.8909866584284965e-4, "batch_size": 64, "dropout": 0.2, "label_smoothing": 0.1},
       "mobilevit_xxs": {"lr": 4.983491647823785e-4, "weight_decay": 6.078408551989326e-4, "batch_size": 32, "dropout": 0.5, "label_smoothing": 0.0}}


class Arr(Dataset):
    def __init__(self, u8, y, tf): self.u8, self.y, self.tf = u8, y, tf
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.tf(Image.fromarray(self.u8[i])), int(self.y[i])


@torch.no_grad()
def evaluate(model, loader, dev):
    model.eval(); ce = nn.CrossEntropyLoss(reduction="sum"); loss, P, L = 0.0, [], []
    for x, y in loader:
        out = model(x.to(dev)); loss += ce(out.float(), y.to(dev)).item(); P.append(out.argmax(1).cpu()); L.append(y)
    p, l = torch.cat(P).numpy(), torch.cat(L).numpy()
    return {"loss": loss / len(l), "acc": float((p == l).mean()), "macro_f1": float(f1_score(l, p, average="macro"))}, p


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True); ap.add_argument("--init", choices=["imagenet", "plantvillage"], required=True)
    ap.add_argument("--ckpt", default=""); ap.add_argument("--seed", type=int, default=0); ap.add_argument("--epochs", type=int, default=30)
    ap.add_argument("--data", default="/kaggle/working/plantdoc_224.npz"); ap.add_argument("--out", default="/kaggle/working")
    ap.add_argument("--workers", type=int, default=2); ap.add_argument("--allow-no-pretrained", action="store_true")
    a = ap.parse_args(); c = CFG[a.model]
    random.seed(a.seed); np.random.seed(a.seed); torch.manual_seed(a.seed)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    d = np.load(a.data); u8, y, split, pv_idx = d["u8"], d["y"], d["split"], d["pv_idx"]; n_cls = len(d["classes"])

    if a.init == "imagenet":
        try:
            model = timm.create_model(a.model, pretrained=True, num_classes=n_cls, drop_rate=c["dropout"])
        except Exception:
            if not a.allow_no_pretrained:
                raise
            model = timm.create_model(a.model, pretrained=False, num_classes=n_cls, drop_rate=c["dropout"])
    else:
        model = timm.create_model(a.model, pretrained=False, num_classes=38, drop_rate=c["dropout"])
        model.load_state_dict(torch.load(a.ckpt, map_location="cpu"))
        head = model.get_classifier(); new = nn.Linear(head.in_features, n_cls)
        with torch.no_grad():
            new.weight.copy_(head.weight[pv_idx]); new.bias.copy_(head.bias[pv_idx])
        model.reset_classifier(n_cls); model.get_classifier().load_state_dict(new.state_dict())
    model.to(dev)

    cfg = timm.models.get_pretrained_cfg(a.model).to_dict(); norm = transforms.Normalize(cfg["mean"], cfg["std"])
    ttf = transforms.Compose([transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(), transforms.RandomRotation(10),
                              transforms.ColorJitter(brightness=(0.5, 1.5)), transforms.ToTensor(), norm])
    etf = transforms.Compose([transforms.ToTensor(), norm])
    kw = dict(num_workers=a.workers, persistent_workers=a.workers > 0)
    tl = DataLoader(Arr(u8[split == "train"], y[split == "train"], ttf), batch_size=c["batch_size"], shuffle=True, drop_last=True, **kw)
    vl = DataLoader(Arr(u8[split == "val"], y[split == "val"], etf), batch_size=128, **kw)
    sl = DataLoader(Arr(u8[split == "test"], y[split == "test"], etf), batch_size=128, **kw)

    opt = torch.optim.AdamW(model.parameters(), lr=c["lr"], weight_decay=c["weight_decay"])
    total, warm = a.epochs * len(tl), len(tl)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    crit = nn.CrossEntropyLoss(label_smoothing=c["label_smoothing"]); scaler = torch.amp.GradScaler("cuda", enabled=dev == "cuda")
    tag = f"plantdoc_{a.model}_{a.init}_seed{a.seed}"; hist, best, t0 = [], None, time.time()
    for ep in range(1, a.epochs + 1):
        model.train(); run, n = 0.0, 0
        for xb, yb in tl:
            xb, yb = xb.to(dev), yb.to(dev); opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", enabled=dev == "cuda"):
                loss = crit(model(xb), yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step(); run += loss.item() * len(yb); n += len(yb)
        val, _ = evaluate(model, vl, dev)
        hist.append({"epoch": ep, "train_loss": run / n, **{f"val_{k}": v for k, v in val.items()}})
        if best is None or val["macro_f1"] > best[0]:   # checkpoint selection on the validation carve-out
            best = (val["macro_f1"], ep, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
    model.load_state_dict(best[2]); test, preds = evaluate(model, sl, dev)
    res = {"tag": tag, "model": a.model, "init": a.init, "seed": a.seed, "epochs": a.epochs, "best_epoch": best[1],
           "val_macro_f1": best[0], "test": test, "train_time_s": time.time() - t0, "history": hist, "test_preds": preds.tolist()}
    json.dump(res, open(f"{a.out}/{tag}.json", "w"))
    print(f"[{tag}] best epoch {best[1]} | test acc {test['acc']:.4f} macro-F1 {test['macro_f1']:.4f} | {res['train_time_s']:.0f}s", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
# ---- Part 2: fine-tuning on PlantDoc (2 models x 2 starting points x 3 seeds), one job per GPU at a time ----
import queue, threading
FT_EPOCHS = 30
jobs = queue.Queue()
for m in RUNS:
    for init in ["imagenet", "plantvillage"]:
        for s in range(3):
            jobs.put((m, init, s))
n_gpus = max(torch.cuda.device_count(), 1); workers = 4 if n_gpus == 1 else 2; failed = []

def gpu_worker(g):
    while True:
        try:
            m, init, s = jobs.get_nowait()
        except queue.Empty:
            return
        if os.path.exists(f"{OUT}/plantdoc_{m}_{init}_seed{s}.json"):
            continue
        cmd = [sys.executable, "-u", f"{OUT}/plantdoc_ft.py", "--model", m, "--init", init, "--seed", str(s),
               "--epochs", str(FT_EPOCHS), "--workers", str(workers)]
        if init == "plantvillage":
            cmd += ["--ckpt", CKPT[(m, str(s))]]
        proc = subprocess.Popen(cmd, env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(g)), stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True)
        for line in proc.stdout:
            if "Warning" not in line and "warn(" not in line:
                print(f"[gpu{g}] {line}", end="", flush=True)
        if proc.wait() != 0:
            failed.append((m, init, s))

threads = [threading.Thread(target=gpu_worker, args=(g,)) for g in range(n_gpus)]
for t in threads: t.start()
for t in threads: t.join()
assert not failed, f"Failed: {failed}"


In [ ]:
# ---- Summary: cross-domain vs. fine-tuned, all on the same PlantDoc test split ----
ft = pd.DataFrame([{k: r[k] for k in ["model", "init", "seed", "best_epoch", "val_macro_f1"]} | {"test_acc": r["test"]["acc"],
                    "test_macro_f1": r["test"]["macro_f1"], "train_min": r["train_time_s"] / 60}
                   for r in (json.load(open(f)) for f in sorted(glob.glob(f"{OUT}/plantdoc_*_seed*.json")))])
ft.to_csv(f"{OUT}/plantdoc_finetune.csv", index=False)
zs_test = zs[(zs.subset == "PlantDoc test split") & zs.precision.str.startswith("fp32")]
table = pd.concat([
    zs_test.assign(setting="PlantVillage only (no PlantDoc training)").rename(columns={"acc": "test_acc", "macro_f1": "test_macro_f1"}),
    ft.assign(setting=ft["init"].map({"imagenet": "Fine-tuned on PlantDoc from ImageNet",
                                      "plantvillage": "Fine-tuned on PlantDoc from our PlantVillage model"}))])
summary = table.groupby(["setting", "model"])[["test_acc", "test_macro_f1"]].agg(["mean", "std"]).round(4)
summary.to_csv(f"{OUT}/plantdoc_summary.csv"); print(summary.to_string())
